![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Quiver Government Contracts Research

This notebook studies whether government contract dollar volume helps explain future returns

In [1]:
qb = QuantBook()
# Anchor the research clock to the start of 2026 for a reproducible history window.
qb.set_start_date(2026, 1, 1)
# Daily bars will have an end_time that matches the following midnight.
qb.settings.daily_precise_end_time = False

### Build a Government Contract Universe

Select assets with frequent, sizable US government contracts, then inspect the returned universe history.

In [2]:
def select_assets(data: List[QuiverGovernmentContractUniverse]) -> List[Symbol]:
    # Group by ticker and keep names with 3+ contracts totalling over $50K.
    contracts_by_symbol: dict[Symbol, list[QuiverGovernmentContractUniverse]] = {}
    for d in data:
        contracts_by_symbol.setdefault(d.symbol, []).append(d)
    return [s for s, ds in contracts_by_symbol.items()
            if len(ds) >= 3 and sum(x.amount or 0 for x in ds) > 50_000]

# Add the Quiver Government Contract universe.
universe = qb.add_universe(QuiverGovernmentContractUniverse, select_assets)
# Request universe history of the last 120 days.
universe_history = qb.universe_history(universe, qb.time - timedelta(120), qb.time - timedelta(1), flatten=True)
# Print the returned shape and columns.
print(f"Shape: {universe_history.shape}")
print(f"Columns: {list(universe_history.columns)}")
universe_history.head()

Shape: (835, 5)
Columns: ['agency', 'amount', 'description', 'value', 'time']


agency  \
time                                                                          
2025-09-04 A RPTMYV3VC57P     National Aeronautics and Space Administration   
           ACH SAG21CX51085                           Department of Defense   
           DNOW VR22RBI5MBFP                General Services Administration   
           EW RTKJ7O7ZTI79                   Department of Veterans Affairs   
           F R735QTJ8XC9X                   General Services Administration   

                                 amount  \
time                                      
2025-09-04 A RPTMYV3VC57P      54940.64   
           ACH SAG21CX51085       34.60   
           DNOW VR22RBI5MBFP     333.44   
           EW RTKJ7O7ZTI79    121000.00   
           F R735QTJ8XC9X      29987.00   

                                                                    description  \
time                                                                              
2025-09-04 A RPTMYV3VC57P                                          AGILENT DEMO   
           ACH SAG21CX51085   4569299007!SHOE POST-OP MEN SM COTTON/POLYESTE...   
           DNOW VR22RBI5MBFP           CANON (CRG-137) TONER CARTRIDGE (2400 YI   
           EW RTKJ7O7ZTI79                                    MONITORING SYSTEM   
           F R735QTJ8XC9X     4X2 PICKUP; COMPACT; CREW CAB;HYBRID ELECTRIC ...   

                                  value time  
time                                          
2025-09-04 A RPTMYV3VC57P      54940.64  NaT  
           ACH SAG21CX51085       34.60  NaT  
           DNOW VR22RBI5MBFP     333.44  NaT  
           EW RTKJ7O7ZTI79    121000.00  NaT  
           F R735QTJ8XC9X      29987.00  NaT

### Universe Diagnostics

Inspects the raw contract amount distribution and visualizes how the daily universe size evolves chronologically.

In [ ]:
# Count selected assets by day.
universe_size = universe_history.groupby(level=[0, 1]).count().amount.groupby(level=0).count()
print(f"Universe days: {len(universe_size)}")
# Store the selected symbol list.
unique_assets = list(universe_history.index.levels[1].unique())
print(f"Mean basket size per day: {universe_size.mean():.1f}")
print('')
print(universe_history.amount.describe().map('{:0.3f}'.format))
universe_size.plot(title='Daily Universe Size', ylabel='Universe Size');

Universe days: 43
Mean basket size per day: 6.2

count         835.000
mean       279986.605
std       2997755.756
min             0.000
25%           338.190
50%          1028.900
75%         14902.700
max      78150743.400
Name: amount, dtype: object


### Daily Universe Prices

Fetch daily price history for every symbol that appears in the universe.

In [ ]:
# Extract unique assets
symbols = list(universe_history.index.get_level_values(1).unique())
# Fetch daily historical price metrics using the earliest timestamp available in the index.
history = qb.history(symbols, universe_history.index[0][0] - timedelta(1), qb.time, Resolution.DAILY)
history

### Align Contract Volume And Returns

Build a joined table of contract dollar volume and future returns.

In [ ]:
dataset = (
    universe_history.groupby(level=[0, 1]).agg(dollarvolume=('amount', 'sum')).rename_axis(['time', 'symbol'])
    .join(history.open.unstack('symbol').sort_index().pct_change(2, fill_method=None).shift(-2).stack().rename('futurereturn').rename_axis(['time','symbol']), how='inner')
)
dataset.head()

### Analyze Relationships Between Factor and Future Returns

Create a scatterplot and plot the line of best fit for the factor.

In [ ]:
y = dataset['futurereturn']
factor = 'dollarvolume'
# Assign the factor values.
x = dataset[factor]
# Drop the 1% outliers on each side of the factor.
lower, upper = x.quantile([0.01, 0.99])
mask = x.between(lower, upper)
x, y = x[mask], y[mask]
# Fit a simple linear model.
slope, intercept = np.polyfit(x, y, 1)
r_squared = x.corr(y) ** 2
# Print the linear model statistics.
print(f"Factor: {factor}")
print(f"Observations: {len(x)}")
print(f"Mean future return: {y.mean():.2%}")
print(f"Alpha: {intercept:.2%}")
print(f"Beta: {slope:.2%}")
print(f"R-squared: {r_squared:.2%}")
# Plot the factor values against future returns.
plt.scatter(x, y, alpha=0.6)
plt.plot(x.sort_values(), intercept + slope * x.sort_values(), color='tab:red', label='Linear fit')
plt.axhline(0, color='black', linewidth=1, alpha=0.4)
plt.title(f'{factor} vs Future Return')
plt.xlabel(factor)
plt.ylabel('Future Return')
plt.legend()
plt.show()

In [ ]:
# Split factor values into quantile buckets.
buckets = pd.qcut(x, q=5, duplicates='drop')
# Summarize each bucket with distribution statistics.
summary = dataset.assign(bucket=buckets).groupby('bucket', observed=True).agg(
    mean_factor=(factor, 'mean'),
    min_future_return=('futurereturn', 'min'),
    max_future_return=('futurereturn', 'max'),
    mean_future_return=('futurereturn', 'mean'),
    std_future_return=('futurereturn', 'std'),
    observations=('futurereturn', 'size')
).reset_index()
summary['bucket'] = summary['bucket'].astype(str)
# Display the bucket summary.
print(f"Factor: {factor}")
display(summary.style.format({
    'mean_factor': '{:.3f}',
    'min_future_return': '{:.2%}',
    'max_future_return': '{:.2%}',
    'mean_future_return': '{:.2%}',
    'std_future_return': '{:.2%}'
}))
# Plot the return distribution for each bucket.
groups = [y[buckets == b].values for b in buckets.cat.categories]
plt.boxplot(groups, labels=[str(b) for b in buckets.cat.categories])
plt.axhline(0, color='black', linewidth=1, alpha=0.4)
plt.title(f'Future Return by {factor} Bucket')
plt.xlabel(f'{factor} Bucket')
plt.ylabel('Future Return')
plt.xticks(rotation=45)
plt.show()